# W08｜Queue / Worker / 长任务恢复｜教学版

**建议学习时间：5h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射完成 Celery/Redis/Worker | 围绕 long-running job | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学 Long-running Agent Task | 知道同步/异步任务边界 | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 做一个最小 background job | 可创建任务并查询状态 | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 复杂分析走 Queue → Worker → Result | HTTP 不再等待长任务 | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 运行中 kill worker | 记录任务会卡在哪里 | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜Background Job API
`POST /analysis → job_id + QUEUED`；`GET /jobs/{id} → RUNNING/SUCCEEDED/FAILED + result`。

### Task 4｜核心链路
`HTTP → create job → queue → worker → persist result → GET result`，HTTP 不等待长分析。

### Task 5｜Kill Worker 参考现象
如果 worker 在标记 RUNNING 后死亡，没有恢复机制时 job 会长期卡 RUNNING；用 heartbeat/lease + reconciliation 将 stale job 转为 RETRYABLE/FAILED。

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 理解 producer/broker/worker
- 设计 Job State Machine
- 理解 crash/recovery

        ## 本周前置检查

        - 完成 W04-W07；有 query_job 表

        ## 本周必须记住的英文术语

        - `broker`：消息代理
- `worker`：工作进程
- `heartbeat`：心跳
- `reconciliation`：恢复对账
- `at-least-once`：至少一次

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. 为什么需要 Queue / Worker？

    **是什么（What）**  
    HTTP 请求适合短交互；长任务应先创建 job，再异步交给 worker。

    **为什么需要（Why）**  
    避免用户连接长时间占用，也便于重试、恢复、限流。

    **什么时候用（When）**  
    长分析、OCR、批量处理、复杂 Agent workflow。

    **常见误区 / 容易踩坑**  
    BackgroundTasks 不是完整任务队列；生产长任务还要考虑持久队列、worker crash、重复执行。


**中英术语**：`producer` = 生产者；`broker` = 消息中间层；`worker` = 工作进程；`job state` = 任务状态；`at-least-once` = 至少一次投递

### 代码实验：最小 Queue / Worker

**这段代码想让你看什么？**

观察 API/producer 只提交 job；worker 在另一个执行上下文处理。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
from queue import Queue
from threading import Thread
import time, uuid

q = Queue()
jobs = {}

def submit_job(payload):
    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "QUEUED", "payload": payload, "result": None}
    q.put(job_id)
    return job_id

def worker():
    while True:
        job_id = q.get()
        if job_id is None:
            q.task_done()
            return
        jobs[job_id]["status"] = "RUNNING"
        time.sleep(0.1)
        jobs[job_id]["result"] = {"rows": 42}
        jobs[job_id]["status"] = "SUCCEEDED"
        q.task_done()

Thread(target=worker, daemon=True).start()
jid = submit_job({"question": "show trend"})
print("submitted:", jobs[jid])
q.join()
print("finished:", jobs[jid])

## 2. Worker Crash 为什么麻烦？

    **是什么（What）**  
    Worker 可能在把 job 标成 RUNNING 后崩溃，导致业务表永远停在 RUNNING。

    **为什么需要（Why）**  
    生产系统必须有 lease/heartbeat/reconciliation/retry。

    **什么时候用（When）**  
    Celery/Redis/RQ/自建 worker 都会遇到。

    **常见误区 / 容易踩坑**  
    不能把“worker 进程存在”当成任务成功；也不能只依赖内存状态。


**中英术语**：`heartbeat` = 心跳；`lease` = 租约；`reconciliation` = 对账/修复；`dead letter` = 失败任务队列

In [ ]:
from datetime import datetime, timedelta

jobs = [
    {"id": "A", "status": "RUNNING", "heartbeat": datetime.now() - timedelta(minutes=30)},
    {"id": "B", "status": "RUNNING", "heartbeat": datetime.now()},
]

cutoff = datetime.now() - timedelta(minutes=5)

for job in jobs:
    if job["status"] == "RUNNING" and job["heartbeat"] < cutoff:
        job["status"] = "RETRYABLE"

print(jobs)

### 练习：设计 Job State Machine

至少包含 QUEUED / RUNNING / SUCCEEDED / FAILED / RETRYABLE。写出哪些状态转换合法。

<details>
    <summary><b>参考答案</b></summary>

    ```python
# 示例
# QUEUED -> RUNNING
# RUNNING -> SUCCEEDED
# RUNNING -> FAILED
# RUNNING -> RETRYABLE   (heartbeat timeout / recoverable failure)
# RETRYABLE -> QUEUED
# SUCCEEDED 不应再次回 RUNNING，除非显式创建新 attempt
```

    </details>

### Checkpoint

1. 为什么 Redis/Celery result backend 不一定适合作为唯一业务 source of truth？
2. Queue 为什么可能重复投递？
3. Worker crash 后最少需要哪些机制？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 业务状态需要长期一致、可审计、可查询，通常应存关系数据库；队列/结果后端主要服务执行。

**2.** 许多消息系统提供 at-least-once 语义，为避免丢消息会允许重复，因此任务必须幂等。

**3.** Heartbeat/lease、stale job reconciliation、retry policy、idempotency、失败记录。

    </details>

## ChatBI 本周作业

        实现：
`POST /analysis -> job_id`
`GET /jobs/{job_id} -> status/result`

再用 Celery + Redis 替换内存 Queue。

### 本周面试 / 理解检查

1. 为什么长任务不建议一直占住 HTTP request？
2. Worker 崩溃后为什么 Job 会卡 RUNNING？
3. Queue 系统为什么仍要 Idempotency？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 容易触发网关/客户端 timeout、占连接，失败恢复和重试也困难。

**2.** 业务状态已写 RUNNING，但没有后续进程完成成功/失败更新。

**3.** 消息可能重复投递或任务在未知状态下被重试，需要避免重复副作用。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。